# One unit per pipeline module

Each section supplies one fixed input, makes a real call into the harness, and shows the relevant output. The normalisation example makes one call through the configured model backend; its evidence is retained in a temporary directory. Q-matrix construction is included because it is the deterministic bridge between items and simulation.

In [2]:
from pathlib import Path
import json
import tempfile
from grammar_kt import canonical, items, kc, kt, normalisation, qmatrix, realisation, simulation, source
from grammar_kt.config import resolve_experiment
from grammar_kt.io import ROOT, read_json, read_jsonl, write_jsonl

settings = resolve_experiment("base").settings

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

## Source

Project one source descriptor onto the five fields visible to phase 1.

In [5]:
source_input = read_jsonl(ROOT / "modules/source/fixtures/core.jsonl")[0]
source_output = source.phase1_record(source_input)
show({"input": source_input, "output": source_output})

{
  "input": {
    "fixture_label": "simple_source_descriptor",
    "egp_id": "FIX_SOURCE_SIMPLE",
    "supercategory": "VERBS",
    "subcategory": "present simple",
    "guideword": "PRESENT SIMPLE",
    "can_do": "Can use the present simple for routines.",
    "examples": [
      "She works every day."
    ]
  },
  "output": {
    "egp_id": "FIX_SOURCE_SIMPLE",
    "supercategory": "VERBS",
    "subcategory": "present simple",
    "guideword": "PRESENT SIMPLE",
    "can_do": "Can use the present simple for routines."
  }
}


## Normalisation

Annotate one fixed EGP descriptor. This calls `normalise_one`, which renders the selected prompt, invokes the configured model, validates the response, and stores the evidence.

In [ ]:
normalisation_input = next(
    row for row in read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl")
    if row["fixture_label"] == "passive"
)
normalisation_evidence = Path(tempfile.mkdtemp(prefix="new-grammar-kt-notebook-normalisation-"))
normalisation_result = normalisation.normalise_one(
    normalisation_input, settings["normalisation"], output=normalisation_evidence
)
show({
    "input": normalisation_input,
    "annotation": normalisation_result["output"],
    "phase2_routing_reason": normalisation_result["phase2_routing_reason"],
    "evidence_directory": normalisation_result["evidence_directory"],
})

{
  "input": {
    "fixture_label": "passive",
    "egp_id": "FIX_NORM_PASSIVE",
    "supercategory": "VERBS",
    "subcategory": "passives",
    "guideword": "PAST SIMPLE PASSIVE",
    "can_do": "Can use the past simple passive in affirmative statements.",
    "examples": [
      "The report was written."
    ]
  },
  "annotation": {
    "egp_id": "FIX_NORM_PASSIVE",
    "result": "complete",
    "cells": [
      {
        "tense": "past",
        "aspect": "none",
        "voice": "passive",
        "polarity": "positive",
        "clause": "declarative",
        "modal": "none"
      }
    ],
    "note": null
  },
  "phase2_routing_reason": "phase1 result complete does not route to phase2",
  "evidence_directory": "/tmp/new-grammar-kt-notebook-normalisation-yyporr6h/units/FIX_NORM_PASSIVE"
}


## Canonical

Turn one complete mapping into a stable canonical cell and its source edge.

In [4]:
canonical_input = {
    "egp_id": "FIX_CANONICAL",
    "result": "complete",
    "cells": [{"tense": "past", "aspect": "none", "voice": "passive",
               "polarity": "positive", "clause": "declarative", "modal": "none"}],
    "note": None,
}
canonical_cells, source_edges = canonical.build([canonical_input])
show({"input": canonical_input, "cells": canonical_cells, "edges": source_edges})

{
  "input": {
    "egp_id": "FIX_CANONICAL",
    "result": "complete",
    "cells": [
      {
        "tense": "past",
        "aspect": "none",
        "voice": "passive",
        "polarity": "positive",
        "clause": "declarative",
        "modal": "none"
      }
    ],
    "note": null
  },
  "cells": [
    {
      "canonical_cell_id": "CELL_B6AF2E896B998C79",
      "cell": {
        "tense": "past",
        "aspect": "none",
        "voice": "passive",
        "polarity": "positive",
        "clause": "declarative",
        "modal": "none"
      },
      "source_descriptor_count": 1,
      "source_edge_count": 1,
      "source_descriptor_ids": [
        "FIX_CANONICAL"
      ],
      "source_mapping_notes": {
        "FIX_CANONICAL": null
      }
    }
  ],
  "edges": [
    {
      "egp_id": "FIX_CANONICAL",
      "source_mapping_result": "complete",
      "source_cell_index": 0,
      "canonical_cell_id": "CELL_B6AF2E896B998C79",
      "source_note": null
    }
  ]
}


## Realisation

Realise and validate one fixed grammar cell/specification pair.

In [5]:
realisation_input = next(
    row for row in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl")
    if row["fixture_label"] == "lexical_present_question"
)
realisation_output = realisation.run_one(realisation_input)
show({"input": realisation_input, "output": realisation_output["output"],
      "valid": realisation_output["valid"], "errors": realisation_output["errors"]})

{
  "input": {
    "fixture_label": "lexical_present_question",
    "cell": {
      "tense": "present",
      "aspect": "none",
      "voice": "active",
      "polarity": "positive",
      "clause": "polar_question",
      "modal": "none"
    },
    "spec": {
      "realization_id": "REAL_0000000000000001",
      "canonical_cell_id": "CELL_0000000000000001",
      "source_descriptor_id": "FIX_REAL_1",
      "predicate_frame_id": "FRAME_WRITE",
      "subject": {
        "text": "the technician",
        "person": 3,
        "number": "singular"
      },
      "wh": null,
      "imperative_subtype": null,
      "let_pronoun": null
    },
    "source_note": null,
    "expected_surface": "Does the technician write the report?"
  },
  "output": {
    "surface": "Does the technician write the report?",
    "auxiliary_chain": [
      "does"
    ],
    "agreement_site": "do",
    "operations": [
      "do_support",
      "operator_inversion"
    ],
    "tokens": [
      "does",
      "the",
 

## KC selection

Apply the factorized policy to one perfect-progressive opportunity.

In [6]:
kc_input = read_json(ROOT / "modules/kc/fixtures/perfect_progressive.json")
kc_result = kc.run_one(kc_input, "factorized")
show({
    "input": kc_input,
    "kc_ids": kc_result["output"]["kc_ids"],
    "activated_rules": [
        {"kc_id": card["kc_id"], "activation_rule": card["activation_rule"]}
        for card in kc_result["kc_specs"]
    ],
})

{
  "input": {
    "fixture_label": "perfect_progressive",
    "opportunity_id": "OPP_FIXTURE_PERFECT_PROGRESSIVE",
    "split": "development",
    "canonical_cell_id": "CELL_FIXTURE_PERFECT_PROGRESSIVE",
    "cell": {
      "tense": "present",
      "aspect": "perfect_progressive",
      "voice": "active",
      "polarity": "positive",
      "clause": "declarative",
      "modal": "none"
    },
    "realization_spec": {
      "realization_id": "REAL_FIXTURE_PERFECT_PROGRESSIVE",
      "canonical_cell_id": "CELL_FIXTURE_PERFECT_PROGRESSIVE",
      "source_descriptor_id": "FIX_KC_PERFECT_PROGRESSIVE",
      "predicate_frame_id": "FRAME_WORK",
      "subject": {
        "text": "the technician",
        "person": 3,
        "number": "singular"
      },
      "wh": null,
      "imperative_subtype": null,
      "let_pronoun": null
    },
    "realization_operations": [
      "perfect_progressive"
    ],
    "source_descriptor_ids": [
      "FIX_KC_PERFECT_PROGRESSIVE"
    ],
    "source_m

## Items

Evaluate one deterministic controlled-transformation item.

In [7]:
item_input = next(
    row for row in read_jsonl(ROOT / "modules/items/fixtures/core.jsonl")
    if row["fixture_label"] == "valid_deterministic_item"
)
item_result = items.evaluate_fixture(item_input)
show({"input": item_input, "realised_answer": item_result["output"]["surface"],
      "valid": item_result["valid"], "errors": item_result["errors"]})

{
  "input": {
    "fixture_label": "valid_deterministic_item",
    "cell": {
      "tense": "past",
      "aspect": "none",
      "voice": "passive",
      "polarity": "positive",
      "clause": "declarative",
      "modal": "none"
    },
    "spec": {
      "realization_id": "REAL_0000000000000011",
      "canonical_cell_id": "CELL_0000000000000011",
      "source_descriptor_id": "FIX_ITEM_1",
      "predicate_frame_id": "FRAME_WRITE",
      "subject": {
        "text": "the report",
        "person": 3,
        "number": "singular"
      },
      "wh": null,
      "imperative_subtype": null,
      "let_pronoun": null
    },
    "target_answer": "The report was written.",
    "accepted_answers": [
      "The report was written."
    ],
    "expected_valid": true
  },
  "realised_answer": "The report was written.",
  "valid": true,
  "errors": []
}


## Q-matrix

Derive the one item–KC edge from a frozen cell projection.

In [8]:
q_item = {"item_id": "ITEM_DEMO", "canonical_cell_id": "CELL_DEMO",
          "all_kc_ids": ["KC_FINITE_PRESENT"],
          "realization_spec": {"realization_id": "REAL_DEMO"},
          "source_descriptor_ids": ["FIX_Q"]}
q_card = {"kc_id": "KC_FINITE_PRESENT",
          "activation_rule": {"cell": {"tense": "present"}}}
q_projection = {"canonical_cell_id": "CELL_DEMO", "kc_ids": ["KC_FINITE_PRESENT"]}
q_columns, q_rows, q_edges, q_audit = qmatrix.build([q_item], [q_card], [q_projection])
show({"input": {"item": q_item, "projection": q_projection},
      "output": {"columns": q_columns, "rows": q_rows, "edges": q_edges, "audit": q_audit}})

{
  "input": {
    "item": {
      "item_id": "ITEM_DEMO",
      "canonical_cell_id": "CELL_DEMO",
      "all_kc_ids": [
        "KC_FINITE_PRESENT"
      ],
      "realization_spec": {
        "realization_id": "REAL_DEMO"
      },
      "source_descriptor_ids": [
        "FIX_Q"
      ]
    },
    "projection": {
      "canonical_cell_id": "CELL_DEMO",
      "kc_ids": [
        "KC_FINITE_PRESENT"
      ]
    }
  },
  "output": {
    "columns": [
      "KC_FINITE_PRESENT"
    ],
    "rows": [
      [
        "ITEM_DEMO",
        [
          1
        ]
      ]
    ],
    "edges": [
      {
        "item_id": "ITEM_DEMO",
        "kc_id": "KC_FINITE_PRESENT",
        "canonical_cell_id": "CELL_DEMO",
        "realization_id": "REAL_DEMO",
        "source_descriptor_ids": [
          "FIX_Q"
        ],
        "activation_rule": {
          "cell": {
            "tense": "present"
          }
        }
      }
    ],
    "audit": {
      "status": "PASS",
      "structural_errors": [],

## Simulation

Simulate the observable history for one fixed learner and the bundled one-item fixture.

In [9]:
simulation_input = {"learner_id": "L0001", "settings": settings["simulation"]}
simulation_result = simulation.run_one(simulation_input["learner_id"], simulation_input["settings"])
show({"input": simulation_input, "output": simulation_result})

{
  "input": {
    "learner_id": "L0001",
    "settings": {
      "parameters": "modules/simulation/configs/default.json",
      "seed": 20260817
    }
  },
  "output": {
    "learner": {
      "learner_id": "L0001",
      "interaction_count": 2,
      "stable_learner": true
    },
    "observable_interactions": [
      {
        "event_id": "EVENT_L0001_001",
        "learner_id": "L0001",
        "item_id": "ITEM_0000000000000001",
        "sequence_index": 1,
        "timestamp": "2026-01-02T00:01:00+00:00",
        "correct": 1,
        "kc_ids": [
          "KC_TENSE_PRESENT"
        ],
        "opportunity_indices": {
          "KC_TENSE_PRESENT": 1
        },
        "canonical_cell_id": "CELL_0000000000000001",
        "item_difficulty": 0.15885412,
        "dataset_split": "train"
      },
      {
        "event_id": "EVENT_L0001_002",
        "learner_id": "L0001",
        "item_id": "ITEM_0000000000000001",
        "sequence_index": 2,
        "timestamp": "2026-01-02T00:02:

## Knowledge tracing

Run the public KT stage on one fixed six-event learner history. Two outcomes in each split keep this unit example small while still permitting AUC calculation.

In [10]:
outcomes = [0, 1, 0, 1, 1, 0]
splits = ["train", "train", "validation", "validation", "test", "test"]
kt_input = [
    {"event_id": f"EVENT_DEMO_{index:03d}", "learner_id": "L_DEMO",
     "item_id": "ITEM_DEMO", "sequence_index": index,
     "timestamp": f"2026-01-01T00:{index:02d}:00+00:00", "correct": outcome,
     "kc_ids": ["KC_FINITE_PRESENT"],
     "opportunity_indices": {"KC_FINITE_PRESENT": index},
     "canonical_cell_id": "CELL_DEMO", "item_difficulty": 0.1,
     "dataset_split": split}
    for index, (outcome, split) in enumerate(zip(outcomes, splits), 1)
]
with tempfile.TemporaryDirectory(prefix="grammar-kt-notebook-kt-") as directory:
    kt_run = Path(directory)
    (kt_run / "simulation").mkdir()
    write_jsonl(kt_run / "simulation/observable_interactions.jsonl", kt_input)
    kt_summary = kt.run(kt_run, {"parameters": "modules/kt/configs/default.json", "techniques": ["empirical", "bkt"]})
    kt_metrics = read_json(kt_run / "kt/metrics.json")
    kt_predictions = read_jsonl(kt_run / "kt/predictions.jsonl")
show({"input": kt_input, "summary": kt_summary,
      "metrics": kt_metrics["techniques"], "predictions": kt_predictions})

{
  "input": [
    {
      "event_id": "EVENT_DEMO_001",
      "learner_id": "L_DEMO",
      "item_id": "ITEM_DEMO",
      "sequence_index": 1,
      "timestamp": "2026-01-01T00:01:00+00:00",
      "correct": 0,
      "kc_ids": [
        "KC_FINITE_PRESENT"
      ],
      "opportunity_indices": {
        "KC_FINITE_PRESENT": 1
      },
      "canonical_cell_id": "CELL_DEMO",
      "item_difficulty": 0.1,
      "dataset_split": "train"
    },
    {
      "event_id": "EVENT_DEMO_002",
      "learner_id": "L_DEMO",
      "item_id": "ITEM_DEMO",
      "sequence_index": 2,
      "timestamp": "2026-01-01T00:02:00+00:00",
      "correct": 1,
      "kc_ids": [
        "KC_FINITE_PRESENT"
      ],
      "opportunity_indices": {
        "KC_FINITE_PRESENT": 2
      },
      "canonical_cell_id": "CELL_DEMO",
      "item_difficulty": 0.1,
      "dataset_split": "train"
    },
    {
      "event_id": "EVENT_DEMO_003",
      "learner_id": "L_DEMO",
      "item_id": "ITEM_DEMO",
      "sequence_index